# Module 9 Exercise (Starter): ViT from scratch vs. CNN baseline vs. fine-tuned pretrained ViT, on CIFAR-10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/09-vit/exercise_starter.ipynb)

Module page: [Module 9: Vision Transformer (ViT)](https://nsteve2407.github.io/llm-transformers-course/modules/09-vit/)

A Vision Transformer treats an image as a sequence of flattened patches and reuses the standard NLP
transformer encoder unchanged -- no convolutions, no built-in locality or translation-equivariance. This
notebook builds one from scratch on CIFAR-10 and puts three models head-to-head:

1. **Data loading** -- real CIFAR-10 (via `torchvision.datasets.CIFAR10`), subsampled for speed, at both
   its native 32x32 resolution (for the from-scratch models) and resized to 224x224 (for the pretrained-ViT
   fine-tuning arm, which requires it).
2. **Patch embedding, two equivalent forms** -- `unfold` + `Linear`, and `Conv2d(kernel=stride=P)` --
   verified numerically identical given matched weights (mirrors Module 2's `Conv2dManual` vs. `F.conv2d`
   check).
3. **ViT-Tiny from scratch**: `[CLS]` token, learned 1D position embeddings, 6 pre-norm encoder blocks
   (4 heads, head dim 48, `d_model=192`), MLP classification head off the final `[CLS]` token.
4. **A comparably-sized CNN baseline**, trained under the same budget.
5. **Both from-scratch models trained** on the CIFAR-10 subset, with loss/accuracy curves.
6. **A pretrained `google/vit-base-patch16-224` fine-tuned** on the same (resized) subset -- real
   ImageNet-pretrained weights, not a random-init stand-in.
7. **A three-way accuracy comparison** of all three models.
8. **Attention rollout**: implemented from scratch (Abnar & Zuidema, 2020) and visualized for correctly-
   and incorrectly-classified examples.
9. A closing discussion of the data-efficiency gap this setup is designed to expose.

Per the course's `SMOKE_TEST` convention, `SMOKE_TEST=1` shrinks the CIFAR-10 subset size and the number of
training epochs so the whole notebook runs quickly end-to-end, but it **always** uses real CIFAR-10 images
(never synthetic/random tensors) and **always** fine-tunes the real pretrained `google/vit-base-patch16-224`
checkpoint -- there is no meaningful "random weights" stand-in for a from-scratch-vs.-pretrained comparison,
the entire point of Part 6/7 is measuring what real pretraining buys you.

In [ ]:
try:
    import transformers
except ImportError:
    %pip install -q transformers

In [ ]:
import glob
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Design choices and judgment calls (documented up front)

- **`SMOKE_TEST` scope**: shrinks the CIFAR-10 subset size (512/128 train/val images vs. 10000/2000 at full
  scale) and the number of training epochs for all three models. It does **not** swap in synthetic data or
  random weights anywhere -- real CIFAR-10 images are loaded and the pretrained ViT arm always downloads and
  fine-tunes real `google/vit-base-patch16-224` weights, since the whole point of this notebook is the
  from-scratch-vs.-pretrained comparison, not just exercising code paths.
- **`P=4` patch size on native 32x32 CIFAR-10 images** gives `N = (32/4)^2 = 64` patches -- a judgment call
  sized to CIFAR-10's small native resolution (contrast with `google/vit-base-patch16-224`'s `P=16` on
  224x224 images, giving `N=196` patches; a 16x16 patch on a 32x32 image would leave only 4 patches, too
  coarse to be a meaningful sequence).
- **`d_model=192`, 4 heads (head dim 48), 6 layers** for "ViT-Tiny" targets roughly 2-3M parameters --
  verified below by actually counting, not just asserted.
- **The CNN baseline is sized to the same order of magnitude, not matched exactly** -- parameter-matching
  CNNs and ViTs exactly is not standard practice and not the point; what matters is that neither model has
  a many-times advantage in capacity, so the comparison isolates architecture/inductive bias rather than
  raw parameter count.
- **224x224 upsampling for the pretrained-ViT arm**: `google/vit-base-patch16-224`'s patch-embedding
  `Conv2d` has a fixed 16x16 kernel/stride tuned to 224x224 inputs, so CIFAR-10's native 32x32 images MUST
  be upsampled via `torchvision.transforms.Resize((224, 224))` before they can be fed in -- a well-known
  practical quirk of fine-tuning ImageNet-pretrained ViTs on small-image datasets, and part of why doing so
  is expensive relative to training directly at native resolution. The 224x224 images are also normalized
  with mean=std=0.5 per channel, matching `google/vit-base-patch16-224`'s own preprocessing config --
  fine-tuning with mismatched normalization statistics measurably hurts accuracy.
- **From-scratch data is preloaded into in-memory tensors** (mirrors Module 2's style; CIFAR-10 at 32x32 is
  small enough this is cheap even at full scale), while the **224x224 pretrained-arm data uses a
  `DataLoader`** over a `Subset` with an on-the-fly resize transform, to avoid holding many-GB of upsampled
  images in memory at full scale.
- **Attention rollout accounts for the residual connection** (Abnar & Zuidema, 2020, *"Quantifying
  Attention Flow in Transformers"*) by averaging attention over heads, mixing in the identity matrix
  (50/50) before each layer's multiplication, and renormalizing -- naively multiplying raw softmax
  attention matrices across layers ignores that every block also passes information through unchanged via
  its residual connection.

## Part 1: data loading -- CIFAR-10, native 32x32 and resized 224x224

In [ ]:
DATA_ROOT = "./data"
N_TRAIN = 512 if SMOKE_TEST else 10000
N_VAL = 128 if SMOKE_TEST else 2000

# --- 32x32 native resolution, preloaded into memory tensors (for the from-scratch ViT-Tiny and CNN) ---
tfm32 = transforms.Compose([transforms.ToTensor()])
train_ds32 = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tfm32)
val_ds32 = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tfm32)

torch.manual_seed(0)
train_idx = torch.randperm(len(train_ds32))[:N_TRAIN]
val_idx = torch.randperm(len(val_ds32))[:N_VAL]

X_train = torch.stack([train_ds32[i][0] for i in train_idx.tolist()])
y_train = torch.tensor([train_ds32[i][1] for i in train_idx.tolist()])
X_val = torch.stack([val_ds32[i][0] for i in val_idx.tolist()])
y_val = torch.tensor([val_ds32[i][1] for i in val_idx.tolist()])

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
print(f"32x32 subset: X_train={tuple(X_train.shape)}, X_val={tuple(X_val.shape)}")

# --- 224x224, required by the pretrained ViT checkpoint's fixed patch-embedding Conv2d -- see the ---
# --- design-choices note above. Loaded via a DataLoader over the SAME subset indices, so both arms ---
# --- of the comparison train/evaluate on identical images, just at different resolutions. Normalized ---
# --- with mean=std=0.5 per channel, matching google/vit-base-patch16-224's own preprocessing config -- ---
# --- fine-tuning with mismatched normalization statistics measurably hurts accuracy, since the ---
# --- pretrained weights expect inputs in roughly the same distribution they were pretrained on. ---
tfm224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])
train_ds224 = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True, transform=tfm224)
val_ds224 = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True, transform=tfm224)

PRETRAINED_BATCH_SIZE = 16 if SMOKE_TEST else 32
train_loader_224 = DataLoader(
    Subset(train_ds224, train_idx.tolist()), batch_size=PRETRAINED_BATCH_SIZE, shuffle=True
)
val_loader_224 = DataLoader(
    Subset(val_ds224, val_idx.tolist()), batch_size=PRETRAINED_BATCH_SIZE, shuffle=False
)
print(
    f"224x224 subset (pretrained-ViT fine-tuning): {len(train_idx)} train / {len(val_idx)} val images, "
    f"batch_size={PRETRAINED_BATCH_SIZE}"
)

## Part 2: patch embedding -- `unfold` + `Linear` vs. `Conv2d(kernel=stride=P)`

A ViT's patch embedding splits an image into non-overlapping `P`x`P` patches and linearly projects each
flattened patch into `d_model` dimensions. There are two equivalent ways to implement this:

- **`unfold` + `Linear`**: `F.unfold` extracts each patch as a flattened `(C*P*P,)` vector (im2col), then a
  single `nn.Linear(C*P*P, d_model)` projects every patch.
- **`Conv2d(kernel_size=P, stride=P)`**: the standard ViT implementation. Because the kernel size equals
  the stride, patches never overlap, and the convolution reduces to exactly the same per-patch linear
  projection -- just computed via a conv kernel instead of an explicit `unfold`.

We implement both and verify numerically that they agree, given matched weights -- mirroring Module 2's
`Conv2dManual` vs. `F.conv2d` validation pattern.

In [ ]:
P = 4  # patch size
IMG = 32  # native CIFAR-10 resolution
N_PATCHES = (IMG // P) ** 2  # 64
D_MODEL = 192


class PatchEmbedUnfold(nn.Module):
    """Patch embedding via F.unfold (im2col) + a single Linear layer applied to every flattened patch."""

    def __init__(self, in_channels=3, patch_size=P, d_model=D_MODEL):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Linear(in_channels * patch_size * patch_size, d_model)

    def forward(self, x):
        # x: (B, C, H, W) -> (B, N_patches, d_model)
        raise NotImplementedError(
            "TODO: use F.unfold(x, kernel_size=self.patch_size, stride=self.patch_size) to extract "
            "flattened patches, transpose to (B, N_patches, C*P*P), then apply self.proj"
        )


class PatchEmbedConv(nn.Module):
    """Patch embedding via Conv2d(kernel_size=stride=patch_size) -- the standard ViT implementation,
    mathematically identical to unfold+Linear for non-overlapping patches (verified below)."""

    def __init__(self, in_channels=3, patch_size=P, d_model=D_MODEL):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: (B, C, H, W) -> (B, N_patches, d_model)
        raise NotImplementedError(
            "TODO: apply self.conv(x) to get (B, d_model, H/P, W/P), then reshape+transpose to "
            "(B, N_patches, d_model)"
        )

In [ ]:
# Validate PatchEmbedUnfold against PatchEmbedConv on a non-trivial random input, with matched weights.
torch.manual_seed(0)
x_probe = torch.randn(4, 3, IMG, IMG)

embed_unfold = PatchEmbedUnfold()
embed_conv = PatchEmbedConv()

# Conv2d's weight (d_model, C, P, P), flattened in (C, kh, kw) order, is exactly nn.Linear's weight
# (d_model, C*P*P) -- F.unfold flattens each patch in that same (C, kh, kw) order by construction, so
# copying one into the other's shape (rather than re-deriving anything) is sufficient to match them.
with torch.no_grad():
    embed_conv.conv.weight.copy_(embed_unfold.proj.weight.reshape(D_MODEL, 3, P, P))
    embed_conv.conv.bias.copy_(embed_unfold.proj.bias)

out_unfold = embed_unfold(x_probe)
out_conv = embed_conv(x_probe)

max_diff = (out_unfold - out_conv).abs().max().item()
print(f"N_PATCHES={N_PATCHES}, output shape={tuple(out_unfold.shape)}, max abs diff={max_diff:.2e}")
assert torch.allclose(out_unfold, out_conv, atol=1e-5), "unfold+Linear and Conv2d patch embeddings disagree!"
print("PASS: unfold+Linear patch embedding matches Conv2d(kernel=stride=P) patch embedding.")

## Part 3: ViT-Tiny from scratch

`[CLS]` token prepended to the patch sequence, learned 1D position embeddings added, 6 pre-norm transformer
encoder blocks (4 heads, head dim 48, `d_model=192` -- same encoder block used in NLP transformers,
unchanged), and an MLP head applied to the final `[CLS]` token's representation.

In [ ]:
N_HEADS = 4
HEAD_DIM = D_MODEL // N_HEADS  # 48
N_LAYERS = 6
MLP_RATIO = 4
N_CLASSES = 10


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x, return_attn=False):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # each (B, n_heads, N, head_dim)
        attn_scores = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn_scores.softmax(dim=-1)  # (B, n_heads, N, N)
        out = attn @ v  # (B, n_heads, N, head_dim)
        out = out.transpose(1, 2).reshape(B, N, D)
        out = self.out_proj(out)
        if return_attn:
            return out, attn
        return out


class EncoderBlock(nn.Module):
    """Pre-norm transformer encoder block: x = x + Attn(LN(x));  x = x + MLP(LN(x))."""

    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, mlp_ratio=MLP_RATIO):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Linear(d_model * mlp_ratio, d_model),
        )

    def forward(self, x, return_attn=False):
        raise NotImplementedError(
            "TODO: implement the pre-norm block: x = x + Attn(LN1(x)); x = x + MLP(LN2(x)). "
            "self.attn(..., return_attn=return_attn) returns (out, attn_weights) when return_attn=True -- "
            "in that case return (x, attn_weights) instead of just x"
        )


class ViTTiny(nn.Module):
    def __init__(self, img_size=IMG, patch_size=P, d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
                 mlp_ratio=MLP_RATIO, n_classes=N_CLASSES):
        super().__init__()
        self.patch_embed = PatchEmbedConv(in_channels=3, patch_size=patch_size, d_model=d_model)
        n_patches = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, mlp_ratio) for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x, return_attn=False):
        B = x.shape[0]
        x = self.patch_embed(x)  # (B, N, D)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, N+1, D)
        x = x + self.pos_embed
        attn_weights_all = []
        for block in self.blocks:
            if return_attn:
                x, attn_w = block(x, return_attn=True)
                attn_weights_all.append(attn_w)
            else:
                x = block(x)
        x = self.ln_final(x)
        logits = self.head(x[:, 0])  # classify off the final [CLS] token
        if return_attn:
            return logits, attn_weights_all
        return logits


torch.manual_seed(0)
model_vit = ViTTiny().to(device)
n_params_vit = sum(p.numel() for p in model_vit.parameters())
print(f"ViT-Tiny: {n_params_vit:,} parameters (target: ~2-3M)")

## Part 4: CNN baseline

A small conv/BN/ReLU stack at increasing channel widths, sized to the same order of magnitude as
ViT-Tiny's parameter count (a documented judgment call -- see the design-choices note above; exact matching
isn't the point).

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=N_CLASSES):
        super().__init__()

        def conv_block(in_c, out_c, stride):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
            )

        self.features = nn.Sequential(
            conv_block(3, 64, stride=1),     # 32x32
            conv_block(64, 128, stride=2),   # 16x16
            conv_block(128, 256, stride=2),  # 8x8
            conv_block(256, 512, stride=2),  # 4x4
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, n_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


torch.manual_seed(0)
model_cnn = SmallCNN().to(device)
n_params_cnn = sum(p.numel() for p in model_cnn.parameters())
print(f"SmallCNN: {n_params_cnn:,} parameters")
print(
    f"ViT-Tiny / SmallCNN parameter ratio: {n_params_vit / n_params_cnn:.2f}x -- same order of magnitude, "
    "not an exact match (documented judgment call above)."
)

## Part 5: train both from-scratch models, and plot loss/accuracy curves

In [ ]:
def train_model(model, X_train, y_train, X_val, y_val, epochs, batch_size, lr):
    """Simple full-batch-in-memory training loop (data already lives on CPU as tensors -- see Part 1)."""
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(X_train.shape[0])
        total_loss, total_correct = 0.0, 0
        for i in range(0, X_train.shape[0], batch_size):
            idx = perm[i : i + batch_size]
            xb, yb = X_train[idx].to(device), y_train[idx].to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * xb.shape[0]
            total_correct += (logits.argmax(dim=-1) == yb).sum().item()
        train_loss = total_loss / X_train.shape[0]
        train_acc = total_correct / X_train.shape[0]

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val.to(device))
            val_loss = F.cross_entropy(val_logits, y_val.to(device)).item()
            val_acc = (val_logits.argmax(dim=-1) == y_val.to(device)).float().mean().item()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(
            f"epoch {epoch + 1}/{epochs}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}"
        )
    return history


EPOCHS_SCRATCH = 2 if SMOKE_TEST else 15
BATCH_SIZE = 32 if SMOKE_TEST else 64
LR = 3e-4
print(f"EPOCHS_SCRATCH={EPOCHS_SCRATCH}, BATCH_SIZE={BATCH_SIZE}, LR={LR}")

In [ ]:
print("Training ViT-Tiny from scratch...")
history_vit = train_model(model_vit, X_train, y_train, X_val, y_val, EPOCHS_SCRATCH, BATCH_SIZE, LR)

print()
print("Training SmallCNN baseline...")
history_cnn = train_model(model_cnn, X_train, y_train, X_val, y_val, EPOCHS_SCRATCH, BATCH_SIZE, LR)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].plot(history_vit["train_loss"], marker="o", label="ViT-Tiny (scratch)")
axes[0].plot(history_cnn["train_loss"], marker="o", label="SmallCNN")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("train loss")
axes[0].set_title("Training loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history_vit["val_acc"], marker="o", label="ViT-Tiny (scratch)")
axes[1].plot(history_cnn["val_acc"], marker="o", label="SmallCNN")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("validation accuracy")
axes[1].set_title("Validation accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: fine-tune a pretrained ViT

`google/vit-base-patch16-224`, loaded via `ViTForImageClassification.from_pretrained(...)` with its
classification head replaced/adapted for CIFAR-10's 10 classes (`num_labels=10,
ignore_mismatched_sizes=True` -- the pretrained head was sized for ImageNet's 1000 classes, so it is
discarded and reinitialized; the backbone's pretrained weights are kept and fine-tuned).

In [ ]:
from transformers import ViTForImageClassification

CHECKPOINT = "google/vit-base-patch16-224"

t0 = time.time()
model_pretrained = ViTForImageClassification.from_pretrained(
    CHECKPOINT, num_labels=N_CLASSES, ignore_mismatched_sizes=True
).to(device)
elapsed = time.time() - t0

print(
    f"Loaded checkpoint '{CHECKPOINT}' in {elapsed:.1f}s "
    f"(config: hidden_size={model_pretrained.config.hidden_size}, "
    f"num_hidden_layers={model_pretrained.config.num_hidden_layers}, "
    f"num_attention_heads={model_pretrained.config.num_attention_heads}) -- this loads real pretrained "
    f"ImageNet weights for the ViT-Base backbone; only the final classification head "
    f"({model_pretrained.config.hidden_size} -> {N_CLASSES}) is newly initialized to match CIFAR-10's classes."
)

n_params_pretrained = sum(p.numel() for p in model_pretrained.parameters())
print(f"pretrained ViT-Base: {n_params_pretrained:,} parameters")

# Concrete evidence the weights were actually downloaded (not a random-init fallback): the on-disk size of
# the Hugging Face cache after loading this checkpoint.
for cache_root in [os.path.expanduser("~/.cache/huggingface"), os.path.expanduser("~/.cache/torch")]:
    if os.path.isdir(cache_root):
        total_bytes = sum(
            os.path.getsize(os.path.join(dp, f))
            for dp, _, files in os.walk(cache_root)
            for f in files
        )
        print(f"cache at {cache_root}: {total_bytes / 1e6:.1f} MB on disk")

In [ ]:
EPOCHS_FINETUNE = 1 if SMOKE_TEST else 3
FINETUNE_LR = 2e-5
print(f"EPOCHS_FINETUNE={EPOCHS_FINETUNE}, FINETUNE_LR={FINETUNE_LR}")

opt_pretrained = torch.optim.AdamW(model_pretrained.parameters(), lr=FINETUNE_LR)
history_pretrained = {"train_loss": [], "val_acc": []}

for epoch in range(EPOCHS_FINETUNE):
    model_pretrained.train()
    total_loss, n_seen = 0.0, 0
    for xb, yb in train_loader_224:
        xb, yb = xb.to(device), yb.to(device)
        opt_pretrained.zero_grad()
        outputs = model_pretrained(pixel_values=xb, labels=yb)
        outputs.loss.backward()
        opt_pretrained.step()
        total_loss += outputs.loss.item() * xb.shape[0]
        n_seen += xb.shape[0]
    train_loss = total_loss / n_seen

    model_pretrained.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader_224:
            xb, yb = xb.to(device), yb.to(device)
            logits = model_pretrained(pixel_values=xb).logits
            correct += (logits.argmax(dim=-1) == yb).sum().item()
            total += xb.shape[0]
    val_acc = correct / total

    history_pretrained["train_loss"].append(train_loss)
    history_pretrained["val_acc"].append(val_acc)
    print(f"epoch {epoch + 1}/{EPOCHS_FINETUNE}  train_loss={train_loss:.4f}  val_acc={val_acc:.3f}")

## Part 7: three-way comparison

Expected pattern (more visible at full scale than smoke scale, where every model gets only a couple of
epochs on a few hundred images): the from-scratch ViT lags the CNN baseline, and the fine-tuned pretrained
ViT wins clearly -- see Part 9 for why.

In [ ]:
final_acc = {
    "ViT-Tiny (scratch)": history_vit["val_acc"][-1],
    "SmallCNN (scratch)": history_cnn["val_acc"][-1],
    "ViT-Base (fine-tuned pretrained)": history_pretrained["val_acc"][-1],
}

print(f"{'model':38s}{'val accuracy':>14s}")
for name, acc in final_acc.items():
    print(f"{name:38s}{acc:14.3f}")

plt.figure(figsize=(6.5, 4.5))
plt.bar(list(final_acc.keys()), list(final_acc.values()), color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("validation accuracy")
plt.title("Final accuracy: from-scratch ViT vs. CNN baseline vs. fine-tuned pretrained ViT")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Part 8: attention rollout

Attention rollout (Abnar & Zuidema, 2020, *"Quantifying Attention Flow in Transformers"*) approximates how
much each input patch's information ultimately reaches the `[CLS]` token's final representation, by
recursively multiplying every layer's (head-averaged) attention matrix together. A raw per-layer attention
matrix alone isn't enough, because each block also passes information through unchanged via its residual
connection -- rollout accounts for this by mixing in the identity matrix before each multiplication.

In [ ]:
def attention_rollout(attn_weights_all):
    """Attention rollout across layers, accounting for the residual connection.

    attn_weights_all: list of length n_layers of (B, n_heads, N, N) post-softmax attention matrices
        (one per encoder block, in order from first to last layer).

    Returns: (B, N, N) rollout matrix. rollout[b, i, j] approximates how much token j's information flows
    into token i's final representation after passing through all layers.
    """
    raise NotImplementedError(
        "TODO: for each layer's (B, n_heads, N, N) attention, average over heads, mix in the identity "
        "matrix 50/50 to account for the residual connection, renormalize rows to sum to 1, then "
        "left-multiply the running rollout by each layer's mixed-and-renormalized matrix in order"
    )

In [ ]:
model_vit.eval()
n_show_pool = min(64, X_val.shape[0])
with torch.no_grad():
    sample_x = X_val[:n_show_pool].to(device)
    sample_y = y_val[:n_show_pool]
    logits, attn_all = model_vit(sample_x, return_attn=True)
    preds = logits.argmax(dim=-1).cpu()

rollout = attention_rollout(attn_all)  # (n_show_pool, N, N), N = N_PATCHES + 1
cls_attn = rollout[:, 0, 1:]  # (n_show_pool, N_PATCHES) -- CLS token's rolled-out attention over patches
grid_size = IMG // P
cls_attn_grid = cls_attn.reshape(-1, grid_size, grid_size)

correct_idx = (preds == sample_y).nonzero(as_tuple=True)[0]
incorrect_idx = (preds != sample_y).nonzero(as_tuple=True)[0]
show_idx = correct_idx[:2].tolist() + incorrect_idx[:2].tolist()
if len(show_idx) < 2:
    show_idx = list(range(min(4, sample_x.shape[0])))
print(f"visualizing {len(show_idx)} examples ({len(correct_idx)} correct / {len(incorrect_idx)} incorrect in pool)")

fig, axes = plt.subplots(2, len(show_idx), figsize=(3 * len(show_idx), 6))
if len(show_idx) == 1:
    axes = axes.reshape(2, 1)

for col, idx in enumerate(show_idx):
    img = sample_x[idx].detach().cpu().permute(1, 2, 0).numpy()
    attn_map = cls_attn_grid[idx].detach().cpu().numpy()
    attn_map_upsampled = np.kron(attn_map, np.ones((P, P)))  # nearest-neighbor: patch grid -> pixel grid
    is_correct = bool(preds[idx] == sample_y[idx])
    title = f"pred={CIFAR10_CLASSES[preds[idx]]}\ntrue={CIFAR10_CLASSES[sample_y[idx]]}"

    axes[0, col].imshow(img)
    axes[0, col].set_title(title, color="green" if is_correct else "red", fontsize=9)
    axes[0, col].axis("off")

    axes[1, col].imshow(img)
    axes[1, col].imshow(attn_map_upsampled, cmap="jet", alpha=0.5)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("input", fontsize=9)
axes[1, 0].set_ylabel("CLS rollout", fontsize=9)
plt.suptitle("Attention rollout: [CLS] token's aggregated attention over input patches")
plt.tight_layout()
plt.show()

## Summary: the data-efficiency gap, and what it demonstrates

- **CNNs bake in locality and translation-equivariance as architectural priors** (a conv filter's weights
  are shared across every spatial position, and a pixel only directly influences nearby outputs). A ViT has
  none of this: patch embedding is the only place position enters the picture in any structured way, and
  every patch attends to every other patch from layer 1 -- the model must *learn* whatever locality
  structure turns out to be useful, rather than getting it for free.
- **That's exactly the trade this notebook is built to expose**: trained from scratch on only a few thousand
  images, ViT-Tiny lacks the inductive bias that lets a similarly-sized CNN generalize efficiently from
  little data, so it should lag SmallCNN's validation accuracy above (more visibly at full scale than smoke
  scale, where both models only see a couple of epochs).
- **Large-scale pretraining is how ViTs buy back that missing inductive bias** -- not by adding architectural
  priors, but by learning, from far more data than any small dataset alone could provide, similar spatial
  regularities that convolutions get for free. That's why the fine-tuned pretrained ViT-Base should win
  clearly over both from-scratch models despite sharing the *same* architecture family (patch embedding +
  encoder blocks + `[CLS]` head) as ViT-Tiny -- pretraining, not architecture, is doing the work.
- **This is the central empirical finding of the original ViT paper** (Dosovitskiy et al., 2021): ViT
  underperforms comparable CNNs when trained from scratch on mid-sized datasets, but matches or exceeds them
  once pretrained on enough data (JFT-300M in the original paper) -- inductive bias substitutes for scale,
  and scale substitutes for inductive bias.
- **Attention rollout gives a window into what the model is actually attending to** -- compare the rolled-out
  attention maps for correctly- vs. incorrectly-classified examples above: does attention concentrate on the
  object, or scatter across background/edges? With only a few epochs of training on a small subset, don't
  expect crisp, object-centered maps -- that's expected at this scale, and part of the same data-efficiency
  story.